# 2. Set up Kubernetes and the shared add-ons (once)

Installs everything that does **not** depend on the VFL method:

1. Base packages on every VM (Docker, kubeadm/kubelet/kubectl).
2. Kubernetes with Kubespray (Flannel over the FABRIC data-plane NIC), run from `control`.
3. Helm, Gateway API CRDs, Linkerd + Jaeger (pinned to `dynamos`), Docker, k9s and an etcd client on `control`.
4. A Python virtual environment on `control` for the experiment runners.

DYNAMOS itself is installed per method in **03_install_method**. Run the cells in order; do not use *Run All*
(the last section is destructive and guarded by a flag).

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run, run_script

fablib = fablib_manager()
slice_ = fh.get_slice(fablib)
nodes = fh.nodes_by_name(slice_)
control = nodes[cfg.CONTROL_NODE]
dynamos = nodes[cfg.DYNAMOS_NODE]
print(f"Slice {slice_.get_name()}: {slice_.get_state()}, lease end {slice_.get_lease_end()}")

## 2.1 Base packages on every VM (in parallel, ~5–10 minutes)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def setup_node(node):
    run_script(node, "node_scripts/node_setup.sh", quiet=True)
    return node.get_name()

with ThreadPoolExecutor(max_workers=len(nodes)) as pool:
    futures = {pool.submit(setup_node, node): name for name, node in nodes.items()}

failed = []
for future, name in futures.items():
    try:
        future.result()
        print(f"{name}: done")
    except Exception as error:
        failed.append(name)
        print(f"{name}: FAILED\n{error}\n")
assert not failed, f"Node setup failed on: {failed}"

## 2.2 Prepare Kubespray on `control`

In [ ]:
run_script(control, "node_scripts/kubespray_setup.sh");

In [ ]:
import os

inventory = fh.kubespray_inventory(slice_)
print(inventory)
os.makedirs("kubespray", exist_ok=True)
with open("kubespray/inventory.ini", "w") as f:
    f.write(inventory)

control.upload_file(local_file_path="kubespray/inventory.ini", remote_file_path="kubespray/inventory/dynamos/inventory.ini")
control.upload_file(local_file_path="kubespray/ansible.cfg", remote_file_path="kubespray/ansible.cfg")
control.upload_file(local_file_path=cfg.SLICE_PRIVATE_KEY, remote_file_path=".ssh/slice_key")
run(control, "chmod 600 ~/.ssh/slice_key", quiet=True);

## 2.3 Install Kubernetes (~15–30 minutes)

In [ ]:
run_script(control, "node_scripts/kubespray_start.sh");

In [ ]:
print(run(control, "kubectl get nodes -o wide", quiet=True))

## 2.4 Shared add-ons on `control`

In [ ]:
run_script(control, "node_scripts/cluster_addons.sh");

## 2.5 Python environment for the experiment runners

In [ ]:
run(control, f"python3 -m venv $HOME/{cfg.EXPERIMENT_VENV} && $HOME/{cfg.EXPERIMENT_VENV}/bin/pip install --upgrade pip", quiet=True)
print(run(control, f"$HOME/{cfg.EXPERIMENT_VENV}/bin/python --version", quiet=True))

## Optional: rebuild Kubernetes from scratch (destructive)

Removes Kubernetes from **all** VMs. Only needed if the cluster is broken; afterwards rerun 2.3 onwards.
Set `RESET_CLUSTER = True` to run it.

In [ ]:
RESET_CLUSTER = False

if RESET_CLUSTER:
    run_script(control, "node_scripts/kubespray_reset.sh")
else:
    print("Skipped (RESET_CLUSTER = False).")